# 2.2.1 读取数据集


In [6]:
# CSV （Comma-Separated Values）文件
# 一个“纯文本版的Excel表格”; 每一行 = 一条数据（row）;逗号, = 分隔列（column）

import os

os.makedirs(os.path.join("..", "data"), exist_ok=True)
# os.makedirs(...) 创建一个目录，如果目录已经存在则不会抛出异常 ；os.path.join("..", "data") 将路径组合成一个完整的路径，这里是上一级目录的data文件夹
data_file = os.path.join("..", "data", "house_tiny.csv")
# 将路径组合成一个完整的路径，这里是上一级目录的data文件夹中的house_tiny.csv文件
with open(data_file, "w") as f:
# with open(...) 打开一个文件，这里是以写入模式("w")打开house_tiny.csv文件，如果文件不存在则会创建它；f是一个文件对象，可以用来写入数据
# with 语句会在代码块执行完毕后自动关闭文件，无论是否发生异常
    f.write("NumRooms,Alley,Price\n")
    f.write("NA,Pave,127500\n")
    f.write("2,NA,106000\n")    # NA = 缺失值  NaN = Not a number
    f.write("4,NA,178100\n")
    f.write("NA,NA,140000\n")

# pandas = 数据分许库；
# CSV → DataFrame（表格结构）
import pandas as pd
data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


# 2.2.2 处理缺失值
1、插值法（用替补值弥补缺失值）
2、删除法（直接忽略缺失值）

In [5]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
# iloc = 按位置取数据（index location）inputs = 特征（X）；outputs = 标签（y）
inputs = inputs.fillna(inputs.mean())
# fillna = “填充 NaN（缺失值）” 用“每一列的平均值”填补缺失值
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)

TypeError: can only concatenate str (not "int") to str

In [ ]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
# iloc = 按位置取数据（index location）inputs = 特征（X）；outputs = 标签（y）

num_cols = inputs.select_dtypes(include='number').columns
cat_cols = inputs.select_dtypes(include='object').columns
# select_dtypes(include='number') → 筛选数值型列（int, float）→ 存入 num_cols ——[num_cols = ['NumRooms'] 
# select_dtypes(include='object') → 筛选字符串/类别型列 → 存入 cat_cols —— cat_cols = ['Alley']  

inputs[num_cols] = inputs[num_cols].fillna(inputs[num_cols].mean()) # fillna = “填充 NaN（缺失值）” 用“每一列的平均值”填补数值型列的缺失值
inputs[cat_cols] = inputs[cat_cols].fillna("Unknown") # fillna = “填充 NaN（缺失值）” 用“Unknown”填补类别型列的缺失值

inputs = pd.get_dummies(inputs, dtype=int) # dtype=float → （0.0/1.0）; dtype=bool  → 布尔 True/False（默认）
# One-Hot 编码 机器学习模型只能处理数字，不能处理文字。get_dummies 就是把文字列转换成数字的工具。1. 找出所有字符串列 2. 统计每列有哪些不同的值 3. 每个值变成一个新列（0/1 或 True/False）
inputs.head() # head() = 显示前几行数据（默认5行）

#

,NumRooms,Alley_Pave,Alley_Unknown
0,3.0,1,0
1,2.0,0,1
2,4.0,0,1
3,3.0,0,1


# 2.2.3  转换为张量格式

In [ ]:
import torch
x = torch.tensor(inputs.to_numpy(dtype = float))  # 把input 的 pandas DataFrame → 转成 numpy 数组，同时把数据类型变成 float
y = torch.tensor(outputs.to_numpy(dtype = float))
x,y                 

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

# 2.3 总结


CSV文件
  ↓ pd.read_csv()   —— 把CSV变成可操作的表格
pandas DataFrame（原始数据）
  ↓ iloc            —— 区分输入X和输出y
拆分特征X 和 标签y
  ↓ select_dtypes
按类型分组列
  ↓ fillna          —— 消除NaN，模型无法处理空值
填充缺失值
  ↓ get_dummies     —— 把文字变成数字
One-Hot编码
  ↓ to_numpy + torch.tensor     —— 变成神经网络可以训练的格式
PyTorch张量（可以训练了）